In [1]:
import sqlite3
import PySimpleGUI as sg
from datetime import datetime, timedelta

# LOGIN USER VARIABLES
login_username = ""
login_user_type = ""


# Database connection
db_path = r"C:/Users/Defne Karaman/OneDrive/Masaüstü/Project_Database 2.db"  # Update with the actual database path
con = sqlite3.connect(db_path)
cur = con.cursor()


# Function to get the next tour ID
def get_next_tour_id(cursor):
    cursor.execute("SELECT MAX(tour_ID) FROM Tour")
    max_id = cursor.fetchone()[0]
    return (max_id + 2) if max_id is not None else 1


# Login Window
def window_login():
    layout = [
        [sg.Text('Welcome to Travelly. Please enter your information.')],
        [sg.Text('Username:', size=(10, 1)), sg.Input(size=(20, 1), key='username')],
        [sg.Text('Password:', size=(10, 1)), sg.Input(size=(20, 1), key='Password', password_char='*')],
        [sg.Button('Login') , sg.Button('Sign up as Customer')]
    ]
    return sg.Window('Login Window', layout)

# Admin Window
def window_admin():
    layout = [
        [sg.Text('Welcome Admin!')],
        [sg.Button('Create a tour')],
        [sg.Button('View tours')],
        [sg.Button('Update Tours')],
        [sg.Button('Logout')]
    ]
    return sg.Window('Admin Window', layout)

# Add Tour Window
def window_add_tour():
    layout = [
        [sg.Text('Tour start date (YYYY-MM-DD):', size=(20, 1)), sg.Input(key='Sdate', size=(20, 1)),
         sg.CalendarButton("Tour Starting Date", close_when_date_chosen=True, target='Sdate', location=(0, 0), format='%Y-%m-%d')],
        [sg.Text('Tour end date (YYYY-MM-DD):', size=(20, 1)), sg.Input(key='Edate', size=(20, 1)),
         sg.CalendarButton("Tour Finishing Date", close_when_date_chosen=True, target='Edate', location=(0, 0), format='%Y-%m-%d')],
        [sg.Text('Price:', size=(20, 1)), sg.Input(size=(20, 1), key='price')],
        [sg.Text('Tour Name:', size=(20, 1)), sg.Input(size=(20, 1), key='tour_name')],
        [sg.Text('Itinerary:', size=(20, 1)), sg.Input(size=(20, 1), key='itinerary')],
        [sg.Text('Maximum Capacity:', size=(20, 1)), sg.Input(size=(20, 1), key='max_cap')],
        [sg.Button('Select Transportation Options'), sg.Button('Add Hotels'), sg.Button('Tour Guide'), sg.Button('Add Tour'), sg.Button('Return to Main Page')]
    ]
    return sg.Window('Add a New Tour', layout)


#TRANSPORTATION
# Retrieve transportation options from the database
def get_transport():
    cur.execute("SELECT transport_ID, transport_type, start_city, destination_city FROM Transport")
    return cur.fetchall()

# Filter transportation options by type, starting city, and destination city
def filter_transportation(options, transport_type, start_city, destination_city):
    filtered = []
    for option in options:
        opt_id, opt_type, opt_start, opt_dest = option
        if transport_type and opt_type.lower() != transport_type.lower():
            continue
        if start_city and opt_start.lower() != start_city.lower():
            continue
        if destination_city and opt_dest.lower() != destination_city.lower():
            continue
        filtered.append(option)
    return filtered

# Transportation Window with Scrollable Layout
def window_add_transport(days):
    transport_layout = []
    for day in range(days):
        transport_layout.append([sg.Text(f"Assign Transportation for Day {day + 1}")])
        transport_layout.append([
            sg.Text('Transport Type:', size=(15, 1)),
            sg.Combo(['Plane', 'Bus', 'Train', 'Cruise', 'Boat'], key=f'transport_type_day_{day + 1}')
        ])
        transport_layout.append([sg.Button('Filter Transport Options', key=f'filter_day_{day + 1}')])
        transport_layout.append([sg.Listbox(values=[], size=(40, 5), key=f'transport_list_day_{day + 1}')])
        transport_layout.append([sg.Button('Select', key=f'select_day_{day + 1}')])

    layout = [
        [sg.Column(transport_layout, scrollable=True, vertical_scroll_only=True, size=(350, 400))],
        [sg.Button('Submit Transportation'), sg.Button('Cancel')]
    ]
    return sg.Window('Assign Transportation', layout)

# Handle transportation assignment
def handle_add_transport(days):
    transport_window = window_add_transport(days)
    all_transport_options = get_transport()
    assigned_transport = {}

    while True:
        event, values = transport_window.read()

        if event == sg.WIN_CLOSED or event == 'Cancel':
            break

        if 'filter_day_' in event:  # Filter transport options for the selected day
            day = int(event.split('_')[-1]) - 1
            transport_type = values[f'transport_type_day_{day + 1}']

            if not transport_type:
                sg.popup("Please select a transport type!")
                continue

            # Filter transport options
            filtered_options = filter_transportation(all_transport_options, transport_type, "", "")
            if filtered_options:
                formatted_results = [f"{t[0]}: {t[1]} -> {t[2]} ({t[3]})" for t in filtered_options]
                transport_window[f'transport_list_day_{day + 1}'].update(values=formatted_results)
            else:
                transport_window[f'transport_list_day_{day + 1}'].update(values=["No options available."])

        if 'select_day_' in event:  # Assign selected transport option to the day
            day = int(event.split('_')[-1]) - 1
            selected_transport = values[f'transport_list_day_{day + 1}']

            if not selected_transport:
                sg.popup("Please select a transport option!")
                continue

            assigned_transport[day + 1] = selected_transport[0]
            sg.popup(f"Transportation for Day {day + 1} assigned successfully!")

        if event == 'Submit Transportation':  # Submit all assigned transportations
            sg.popup(f"Transportation assigned: {assigned_transport}")
            break

    transport_window.close()
    return assigned_transport


#HOTEL
# Retrieve hotels by city from the database
def get_hotels_by_city(city):
    cur.execute("SELECT hotel_ID, hotel_name, city FROM Hotel WHERE city = ?", (city,))
    return cur.fetchall()

# Hotel Assignment Window
def window_add_hotel():
    layout = [
        [sg.Text("Enter City:", size=(15, 1)), sg.Input(size=(20, 1), key="CITY")],
        [sg.Button("Filter")],
        [sg.Text("Available Hotels:")],
        [sg.Listbox(values=[], size=(50, 10), key="HOTEL_LIST")],
        [sg.Button("Select Hotel"), sg.Button("Cancel"), sg.Button("Submit")]
    ]
    return sg.Window('Assign Hotels', layout)

# Handle hotel assignment
def handle_add_hotel():
    hotel_window = window_add_hotel()
    assigned_hotels = []

    while True:
        event, values = hotel_window.read()

        if event == sg.WIN_CLOSED or event == 'Cancel':
            break

        if event == "Filter":
            city = values["CITY"].strip()

            if not city:
                sg.popup("Please enter a city name!")
                continue

            # Fetch hotels for the entered city
            filtered_hotels = get_hotels_by_city(city)
            if filtered_hotels:
                formatted_results = [f"{h[0]}: {h[1]} ({h[2]})" for h in filtered_hotels]
                hotel_window["HOTEL_LIST"].update(values=formatted_results)
            else:
                hotel_window["HOTEL_LIST"].update(values=["No hotels available in this city."])

        if event == "Select Hotel":
            selected_hotel = values["HOTEL_LIST"]

            if not selected_hotel:
                sg.popup("Please select a hotel from the list!")
                continue

            hotel_id = selected_hotel[0].split(":")[0]
            if hotel_id.isdigit():
                assigned_hotels.append(selected_hotel[0])
                sg.popup(f"Selected Hotel: {selected_hotel[0]}")
            else:
                sg.popup("Invalid hotel selection!")

        if event == "Submit":
            if not assigned_hotels:
                sg.popup("You must select at least one hotel!")
            else:
                sg.popup("Hotels assigned successfully!")
            break

    hotel_window.close()
    return assigned_hotels


# Display Tours
def get_tours():
    try:
        cur.execute("SELECT tour_ID, tour_name, Sdate, Edate, price, max_cap, current_cap FROM Tour")
        tours = cur.fetchall()
        if not tours:
            return ["No tours available."]
        return [
            f"ID: {tour[0]} | Name: {tour[1]} | Start: {tour[2]} | End: {tour[3]} | Price: {tour[4]} | Max Capacity: {tour[5]}  | Current Capacity: {tour[6]}"
            for tour in tours
        ]
    except sqlite3.Error as e:
        return [f"Error fetching tours: {e}"]

def window_view_tours():
    tours = get_tours()  # Fetch tours from the database
    layout = [
        [sg.Text("Available Tours:")],
        [sg.Listbox(values=tours, size=(80, 20), key="TOUR_LIST", enable_events=False)],
        [sg.Button("Back to Main Page")]
    ]
    return sg.Window('View Tours', layout)


# Assign a tour guide to a tour
def window_add_tour_guides(start_date, end_date):
    if not start_date or not end_date:
        sg.popup("Please select the tour start and end dates first!")
        return None

    try:
        # available guides
        cur.execute(
            '''SELECT TG_username FROM TourGuide tg
               WHERE TG_username NOT IN (
                   SELECT TG_username FROM Assign
                   WHERE tour_ID IN (
                       SELECT tour_ID FROM Tour
                       WHERE NOT (Edate < ? OR Sdate > ?)
                   )
               )''',
            (start_date, end_date)
        )
        available_guides = cur.fetchall()
        if not available_guides:
            sg.popup("No tour guides are available for the selected dates.")
            return None

        # guide options
        guide_options = [guide[0] for guide in available_guides]
        layout = [
            [sg.Text("Select Tour Guides:")],
            [sg.Listbox(values=guide_options, size=(50, 10), key="GUIDE_LIST", select_mode='multiple')],
            [sg.Button('Submit'), sg.Button('Cancel')]
        ]
        guide_window = sg.Window('Assign Tour Guides', layout)

        selected_guides = []
        while True:
            event, values = guide_window.read()
            if event == sg.WIN_CLOSED or event == 'Cancel':
                break
            if event == 'Submit':
                selected_guides = values['GUIDE_LIST']
                if not selected_guides:
                    sg.popup("You must select at least one tour guide!")
                else:
                    sg.popup("Tour guides assigned successfully!")
                    break

        guide_window.close()
        return selected_guides

    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")
        return None


# Save tour and transit data to the database
def button_add_tour(window, values, assigned_transport, assigned_hotels, assigned_guides):
    # Debug logs checkpoint
    print(f"Assigned Guides: {assigned_guides}") 
    print(f"Assigned Transport: {assigned_transport}")  
    print(f"Assigned Hotels: {assigned_hotels}") 

    required_fields = ['Sdate', 'Edate', 'price', 'tour_name', 'itinerary', 'max_cap']
    missing_fields = [field for field in required_fields if not values[field]]

    if missing_fields:
        sg.popup(f"These fields cannot be empty: {', '.join(missing_fields)}")
        return

    if not assigned_transport or not isinstance(assigned_transport, dict) or len(assigned_transport) == 0:
        sg.popup("Transportation details are required!")
        return

    if not assigned_hotels or not isinstance(assigned_hotels, list) or len(assigned_hotels) == 0:
        sg.popup("Hotel details are required!")
        return

    if not assigned_guides or not isinstance(assigned_guides, list) or len(assigned_guides) == 0:
        sg.popup("Tour Guide details are required!")
        return

    try:
        start_date = datetime.strptime(values['Sdate'], '%Y-%m-%d')
        end_date = datetime.strptime(values['Edate'], '%Y-%m-%d')

        if end_date <= start_date:
            sg.popup('End date must be after the start date!')
            return

        # Insert into the Tour table
        new_tour_id = get_next_tour_id(cur)
        cur.execute(
            '''INSERT INTO Tour (tour_ID, tour_name, Sdate, Edate, price, itinerary, max_cap, current_cap) 
               VALUES (?, ?, ?, ?, ?, ?, ?, 0)''',
            (new_tour_id, values['tour_name'], values['Sdate'], values['Edate'], float(values['price']),
             values['itinerary'], int(values['max_cap']))
        )

        # Insert assigned transportation into Transit table
        if assigned_transport:
            for day, transport in assigned_transport.items():
                transport_id = transport.split(":")[0]  # Extract transport ID
                cur.execute(
                    '''INSERT INTO Transit (transport_ID, tour_ID, day)
                       VALUES (?, ?, ?)''',
                    (transport_id, new_tour_id, day)
                )

        # Insert assigned hotels into Accommodate table
        if assigned_hotels:
            for hotel in assigned_hotels:
                hotel_id = hotel.split(":")[0]
                if hotel_id.isdigit():
                    # Use the tour's start date as the default date for the Accommodate table
                    accomodate_date = values['Sdate']
                    cur.execute(
                        '''INSERT INTO Accomodate (hotel_ID, date, tour_ID)
                           VALUES (?, ?, ?)''',
                        (hotel_id, accomodate_date, new_tour_id)
                    )
                else:
                    sg.popup_error(f"Invalid hotel ID: {hotel}")

        # Insert assigned guides into Assign table
        if assigned_guides:
            for guide in assigned_guides:
                cur.execute(
                    '''INSERT INTO Assign (TG_username, tour_ID)
                       VALUES (?, ?)''',
                    (guide, new_tour_id)
                )

        # Explicitly commit the changes
        con.commit()

        sg.popup("Tour, transportation, hotel, and guide assignments successfully added!")

    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")
    except Exception as e:
        sg.popup_error(f"Unexpected error: {e}")

    # Transition back to main page
    window.close()
    return window_admin()


#Tour Guide Login
def window_guide_login():
    layout = [
        [sg.Text('Tour Guide Login')],
        [sg.Text('Username:', size=(10, 1)), sg.Input(size=(20, 1), key='guide_username')],
        [sg.Text('Password:', size=(10, 1)), sg.Input(size=(20, 1), key='guide_password', password_char='*')],
        [sg.Button('Login as Tour Guide'), sg.Button('Back to Main Page')]
    ]
    return sg.Window('Tour Guide Login', layout)

# Display Tour Guide's scheduled tours
def get_guide_scheduled_tours(guide_username):
    try:
        cur.execute(
            '''
            SELECT t.tour_ID, t.tour_name, t.Sdate, t.Edate, t.price, t.max_cap, t.current_cap
            FROM Tour t
            JOIN Assign a ON t.tour_ID = a.tour_ID
            WHERE a.TG_username = ?
            ''',
            (guide_username,)
        )
        tours = cur.fetchall()
        if not tours:
            return ["No tours assigned."]
        return [
            f"ID: {tour[0]} | Name: {tour[1]} | Start: {tour[2]} | End: {tour[3]} | Price: {tour[4]} | Capacity: {tour[5]}"
            for tour in tours
        ]
    except sqlite3.Error as e:
        return [f"Error fetching tours: {e}"]

def window_guide_tours(guide_username):
    # Fetch scheduled tours
    scheduled_tours = get_guide_scheduled_tours(guide_username)
    
    # Fetch the fixed salary of the tour guide
    try:
        cur.execute('SELECT salary FROM TourGuide WHERE TG_username = ?', (guide_username,))
        salary_result = cur.fetchone()
        if salary_result: 
            salary = f"Fixed Salary: ${salary_result[0]}" 
        else:
            return ["Salary information not available."]
            
    except sqlite3.Error as e:
        salary = f"Error fetching salary: {e}"

    # Adjusted layout
    layout = [
        [sg.Text('Scheduled Tours:')],
        [sg.Listbox(values=scheduled_tours, size=(60, 15), key='GUIDE_TOUR_LIST', enable_events=False)],  # Adjusted size
        [sg.Text(salary)],  # Display salary
        [sg.Button('Customer Information'), sg.Button('Logout')]
    ]
    return sg.Window('Tour Guide Scheduled Tours', layout)

def get_customer_information(tour_id):
    try:
        cur.execute(
            '''
            SELECT u.Name, u.Sname, t.contact_no
            FROM User u
            JOIN Traveler t ON u.Username = t.T_username
            JOIN Purchase p ON t.T_username = p.T_username
            WHERE p.tour_ID = ?
            ''',
            (tour_id,)
        )
        customers = cur.fetchall()
        if not customers:
            return ['No customers have purchased this tour.']
        return [
            f"Name: {customer[0]} {customer[1]} | Contact: {customer[2]}"
            for customer in customers
        ]
    except sqlite3.Error as e:
        return [f"Error fetching customer information: {e}"]

def window_customer_information(customer_info):
    layout = [
        [sg.Text('Customer Information:')],
        [sg.Listbox(values=customer_info, size=(80, 20), key='CUSTOMER_INFO_LIST')],
        [sg.Button('Back')]
    ]
    return sg.Window('Customer Information', layout)

In [2]:
#Second Requirement
# Customer Sign Up
def window_sign_up():
    layout = [
        [sg.Text('Sign Up')],
        [sg.Text('Username:', size=(15, 1)), sg.Input(size=(20, 1), key='new_username')],
        [sg.Text('Password:', size=(15, 1)), sg.Input(size=(20, 1), key='new_password', password_char='*')],
        [sg.Text('Name:', size=(15, 1)), sg.Input(size=(20, 1), key='name')],
        [sg.Text('Surname:', size=(15, 1)), sg.Input(size=(20, 1), key='surname')],
        [sg.Button('Register'), sg.Button('Cancel')]
    ]
    return sg.Window('Customer Sign Up', layout)

def window_customer_profile():
    layout = [
        [sg.Text(f"Welcome, {login_username}! This is your profile.")],
        [sg.Button('Add Information')],
        [sg.Button('View Paid Tours')],
        [sg.Button('Tour Reviews')],
        [sg.Button('Purchase Tours')],
        [sg.Button('Logout')]
    ]
    return sg.Window('Customer Profile', layout)

def handle_sign_up(window, values):
    username, password = values['new_username'], values['new_password']
    name, surname = values['name'], values['surname']
    if not username or not password or not name or not surname:
        sg.popup('All fields are required!')
        return None

    try:
        # Check if the username already exists explicitly
        cur.execute("SELECT Username FROM User WHERE Username = ?", (username,))
        if cur.fetchone():
            sg.popup("Username already exists. Choose a different username.")
            return None

        cur.execute(
            '''INSERT INTO User (Username, Name, Sname, password)
               VALUES (?, ?, ?, ?)''',
            (username, name, surname, password)
        )

        cur.execute(
            '''INSERT INTO Traveler (T_username, credit_card_no, credit_card_expiry_date, 
               credit_card_holder_name, billing_address, contact_no)
               VALUES (?, '', '1970-01-01', '', '', '')''', 
            (username,)
        )
        con.commit()
        sg.popup('Sign-up successful! Please log in.')
        window.close()
        return window_login()
    
    except sqlite3.IntegrityError as e:
        print(f"IntegrityError: {e}")  # Log the error for debugging
        sg.popup('Username already exists or another database constraint was violated.')
        return None
    except sqlite3.Error as e:
        print(f"Database Error: {e}")  # Log general database errors
        sg.popup(f"An unexpected database error occurred: {e}")
        return None

# Customer Profile
def handle_profile_update(values):
    contact_no = values['contact']
    credit_card_no = values['credit_card']
    if not contact_no or not credit_card_no:
        sg.popup('Both fields are required!')
        return

    try:
        cur.execute(
            '''UPDATE Traveler
               SET contact_no = ?, credit_card_no = ?
               WHERE T_username = ?''',
            (contact_no, credit_card_no, login_username)
        )
        con.commit()
        sg.popup('Profile updated successfully!')
    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")

def window_add_information():
    layout = [
        [sg.Text("What information would you like to add?")],
        [sg.Button('Add Credit Card Information')],
        [sg.Button('Add Contact No')],
        [sg.Button('Back')]
    ]
    return sg.Window('Add Information', layout)

def window_add_credit_card():
    layout = [
        [sg.Text("Enter Credit Card Information")],
        [sg.Text('Card Number:', size=(20, 1)), sg.Input(size=(20, 1), key='credit_card_no')],
        [sg.Text('Expiry Date (YYYY-MM-DD):', size=(20, 1)), sg.Input(size=(20, 1), key='credit_card_expiry_date')],
        [sg.Text('Cardholder Name:', size=(20, 1)), sg.Input(size=(20, 1), key='credit_card_holder_name')],
        [sg.Text('Billing Address:', size=(20, 1)), sg.Input(size=(20, 1), key='billing_address')],
        [sg.Button('Save'), sg.Button('Go Back')]
    ]
    return sg.Window('Add Credit Card Information', layout)

def handle_add_credit_card(values):
    credit_card_no = values['credit_card_no']
    expiry_date = values['credit_card_expiry_date']
    cardholder_name = values['credit_card_holder_name']
    billing_address = values['billing_address']

    if not all([credit_card_no, expiry_date, cardholder_name, billing_address]):
        sg.popup('All fields are required!')
        return

    try:
        cur.execute(
            '''UPDATE Traveler
               SET credit_card_no = ?, credit_card_expiry_date = ?, credit_card_holder_name = ?, billing_address = ?
               WHERE T_username = ?''',
            (credit_card_no, expiry_date, cardholder_name, billing_address, login_username)
        )
        con.commit()
        sg.popup('Credit card information added successfully!')
    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")

def add_contact_no_popup():
    contact_no = sg.popup_get_text("Enter your contact number:")
    if not contact_no:
        sg.popup("No contact number entered!")
        return

    try:
        cur.execute(
            '''UPDATE Traveler
               SET contact_no = ?
               WHERE T_username = ?''',
            (contact_no, login_username)
        )
        con.commit()
        sg.popup("Contact number added successfully!")
    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")

# Tour Search Page
def search_tours(values):
    start_date, end_date = values['start_date'], values['end_date']
    try:
        # Fetch tours not started yet and within date range
        cur.execute(
            '''SELECT tour_ID, tour_name, Sdate, Edate, price, max_cap
               FROM Tour
               WHERE Sdate >= ? AND Edate <= ?''',
            (start_date, end_date)
        )
        tours = cur.fetchall()
        if not tours:
            return ["No tours available."]
        return [
            f"ID: {tour[0]} | Name: {tour[1]} | Start: {tour[2]} | End: {tour[3]} | Price: {tour[4]} | Capacity: {tour[5]}"
            for tour in tours
        ]
    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")
        return []

def get_upcoming_tours(start_date=None, end_date=None):
    query = '''SELECT tour_ID, tour_name, Sdate, Edate, price, max_cap, current_cap
               FROM Tour
               WHERE Sdate > ? AND max_cap > 0'''
    params = [datetime.now().strftime('%Y-%m-%d')]

    if start_date:
        query += " AND Sdate >= ?"
        params.append(start_date)
    if end_date:
        query += " AND Edate <= ?"
        params.append(end_date)

    try:
        cur.execute(query, tuple(params))
        tours = cur.fetchall()
        if not tours:
            return ["No tours available."]
        return [
            f"ID: {tour[0]} | Name: {tour[1]} | Start: {tour[2]} | End: {tour[3]} | Price: {tour[4]} | Max Capacity: {tour[5]} | Current Capacity: {tour[6]}"
            for tour in tours
        ]
    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")
        return []

def window_tour_details_purchase(tour_id):
    try:
        # Fetch tour details
        cur.execute(
            '''SELECT t.tour_name, t.Sdate, t.Edate, t.price, t.itinerary, t.max_cap, t.current_cap
               FROM Tour t
               WHERE t.tour_ID = ?''',
            (tour_id,)
        )
        tour = cur.fetchone()

        # Fetch associated transportation
        cur.execute(
            '''SELECT transport_type FROM Transport tr
               JOIN Transit ts ON tr.transport_ID = ts.transport_ID
               WHERE ts.tour_ID = ?''',
            (tour_id,)
        )
        transport = [t[0] for t in cur.fetchall()]

        # Fetch associated accommodations
        cur.execute(
            '''SELECT hotel_name FROM Hotel h
               JOIN Accomodate a ON h.hotel_ID = a.hotel_ID
               WHERE a.tour_ID = ?''',
            (tour_id,)
        )
        accommodations = [a[0] for a in cur.fetchall()]

        # Fetch assigned guides
        cur.execute(
            '''SELECT TG_username FROM Assign WHERE tour_ID = ?''',
            (tour_id,)
        )
        guides = [g[0] for g in cur.fetchall()]

        layout = [
            [sg.Text(f"Tour Name: {tour[0]}")],
            [sg.Text(f"Start Date: {tour[1]}")],
            [sg.Text(f"End Date: {tour[2]}")],
            [sg.Text(f"Price: {tour[3]}")],
            [sg.Text(f"Itinerary: {tour[4]}")],
            [sg.Text(f"Remaining Capacity: {tour[5]}")],
            [sg.Text("Transportation Options:")],
            [sg.Text(", ".join(transport))],
            [sg.Text("Accommodations:")],
            [sg.Text(", ".join(accommodations))],
            [sg.Text("Tour Guides:")],
            [sg.Text(", ".join(guides))],
            [sg.Button('Back')]
        ]
        return sg.Window('Tour Details', layout)

    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")
        return None

# Select and Pay 
def window_purchase_tours():
    upcoming_tours = get_upcoming_tours()
    layout = [
        [sg.Text('Search Available Tours')],
        [sg.Text('Tour start date (YYYY-MM-DD):', size=(20, 1)), sg.Input(key='Sdate', size= (20,1)),
         sg.CalendarButton("Tour Starting Date",close_when_date_chosen=True, target='Sdate', location=(0,0), format='%Y-%m-%d', no_titlebar=False)],
        [sg.Text('Tour end date (YYYY-MM-DD):', size=(20, 1)), sg.Input(key='Edate', size=(20,1)),
         sg.CalendarButton("Tour Finishing Date",close_when_date_chosen=True, target='Edate', location=(0,0), format='%Y-%m-%d', no_titlebar=False)],
        [sg.Button('Search')],
        [sg.Text('Available Tours:')],
        [sg.Listbox(values=upcoming_tours, size=(80, 20), key='TOUR_LIST', enable_events=True)],
        [sg.Button('View Details'), sg.Button('Pay for Tour'), sg.Button('Back')]
    ]

    window = sg.Window('Purchase Tours', layout)
    selected_tour_id = None

    while True:
        event, values = window.read()

        if event == sg.WIN_CLOSED or event == 'Back':
            window.close()
            return window_customer_profile()

        if event == 'Search':
            start_date = values.get('Sdate')
            end_date = values.get('Edate')
            if not validate_dates(start_date, end_date):
                continue
            tours = get_upcoming_tours(start_date, end_date)
            window['TOUR_LIST'].update(tours)

        if event == 'TOUR_LIST':
            selected_tour = values['TOUR_LIST']
            if selected_tour and "ID:" in selected_tour[0]:
                selected_tour_id = selected_tour[0].split('|')[0].strip().replace('ID: ', '')

        if event == 'View Details':
            try:
                if selected_tour_id:
                    details_window = window_tour_details_purchase(selected_tour_id)
                    while True:
                        detail_event, _ = details_window.read()
                        if detail_event in (sg.WIN_CLOSED, 'Back'):
                            details_window.close()
                            break
                else:
                    sg.popup("Please select a tour from the list.")
            except NameError:
                sg.popup("No tour selected. Please select a tour to view details.")

        if event == 'Pay for Tour':
            try:
                if selected_tour_id:
                    if pay_for_tour(selected_tour_id):
                        start_date = values.get('Sdate')
                        end_date = values.get('Edate')
                        window['TOUR_LIST'].update(get_upcoming_tours(start_date, end_date))
                else:
                    sg.popup("Please select a tour from the list.")
            except NameError:
                sg.popup("No tour selected. Please select a tour to pay for.")

    window.close()

def pay_for_tour(tour_id):
    try:
        # Check if the tour exists and has capacity
        cur.execute(
            '''SELECT current_cap FROM Tour WHERE tour_ID = ? AND current_cap < max_cap''',
            (tour_id,)
        )
        tour = cur.fetchone()
        if not tour:
            sg.popup('Tour is fully booked!')
            return False

        # Check if the customer has already booked this tour
        cur.execute(
            '''SELECT * FROM Purchase WHERE tour_ID = ? AND T_username = ?''',
            (tour_id, login_username)
        )
        if cur.fetchone():
            sg.popup('You have already booked this tour!')
            return False

        # Ensure the customer has credit card information
        cur.execute(
            '''SELECT credit_card_no FROM Traveler WHERE T_username = ?''',
            (login_username,)
        )
        credit_card = cur.fetchone()
        if not credit_card or not credit_card[0]:
            # Show popup and guide customer to add credit card details
            sg.popup(
                'Credit Card Information Missing',
                'You need to add your credit card information before purchasing a tour.',
                'Please update your profile with valid credit card information.'
            )
            return False

        # Update Tour capacity
        cur.execute(
            '''UPDATE Tour
               SET current_cap = current_cap + 1
               WHERE tour_ID = ?''',
            (tour_id,)
        )

        # Insert into Purchase table
        cur.execute(
            '''INSERT INTO Purchase (tour_ID, T_username)
               VALUES (?, ?)''',
            (tour_id, login_username)
        )

        # Commit changes and show success popup
        con.commit()
        sg.popup('Payment successful!')
        return True

    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")
        return False

# My Tours Page
def get_my_tours():
    try:
        cur.execute(
            '''SELECT t.tour_ID, t.tour_name, t.Sdate, t.Edate, t.price, t.itinerary
               FROM Tour t
               JOIN Purchase p ON t.tour_ID = p.tour_ID
               WHERE p.T_username = ?''',
            (login_username,)
        )
        tours = cur.fetchall()
        if not tours:
            return ["No tours purchased."]
        return [
            f"ID: {tour[0]} | Name: {tour[1]} | Start: {tour[2]} | End: {tour[3]} | Price: {tour[4]} | Itinerary: {tour[5]}"
            for tour in tours
        ]
    except sqlite3.Error as e:
        return [f"Error fetching tours: {e}"]

def window_view_paid_tours():
    my_tours = get_my_tours()  # Fetch paid tours
    layout = [
        [sg.Text("Your Paid Tours:")],
        [sg.Listbox(values=my_tours, size=(80, 20), key='PAID_TOURS')],
        [sg.Button('Back')]
    ]
    return sg.Window('Paid Tours', layout)

# Date validation and error
def validate_dates(start_date, end_date):
    try:
        start_date_obj = datetime.strptime(start_date, '%Y-%m-%d')
        end_date_obj = datetime.strptime(end_date, '%Y-%m-%d')
        if start_date_obj > end_date_obj:
            sg.popup_error("Invalid date range: Start date cannot be later than end date.")
            return False
        return True
    except ValueError:
        sg.popup_error("Invalid date format. Please enter dates in YYYY-MM-DD format.")
        return False  
 

In [3]:
# Third requirement

# Get past and upcoming tours for the logged-in customer
def get_customer_tours():
    try:
        today = datetime.now().strftime('%Y-%m-%d')
        
        # Fetch past tours
        cur.execute(
            '''
            SELECT t.tour_ID, t.tour_name, t.Sdate, t.Edate, t.price, t.itinerary
            FROM Tour t
            JOIN Purchase p ON t.tour_ID = p.tour_ID
            WHERE p.T_username = ? AND t.Edate < ?
            ''',
            (login_username, today)
        )
        past_tours = cur.fetchall()

        # Fetch upcoming tours
        cur.execute(
            '''
            SELECT t.tour_ID, t.tour_name, t.Sdate, t.Edate, t.price, t.itinerary
            FROM Tour t
            JOIN Purchase p ON t.tour_ID = p.tour_ID
            WHERE p.T_username = ? AND t.Sdate >= ?
            ''',
            (login_username, today)
        )
        upcoming_tours = cur.fetchall()

        return past_tours, upcoming_tours

    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")
        return [], []

# Window to leave a review
def window_leave_review(tour_id, tour_name):
    layout = [
        [sg.Text(f"Leave a Review for: {tour_name}")],
        [sg.Text("Rating (1-5):", size=(15, 1)), sg.Combo([1, 2, 3, 4, 5], key='rating')],
        [sg.Text("Review:", size=(15, 1)), sg.Multiline(size=(40, 5), key='review_text')],
        [sg.Button("Submit"), sg.Button("Cancel")]
    ]
    return sg.Window("Leave a Review", layout)

def get_next_review_id():
    try:
        cur.execute("SELECT MAX(review_ID) FROM HasReview")
        max_id = cur.fetchone()[0]
        return (max_id + 1) if max_id is not None else 1
    except sqlite3.Error as e:
        sg.popup_error(f"Database error while generating review ID: {e}")
        return None

# Handle the review submission
def handle_leave_review(tour_id, values):
    rating = values.get("rating")
    review = values.get("review_text")
    today = datetime.now().strftime('%Y-%m-%d')

    if not rating or not review.strip():
        sg.popup("Please provide both a rating and a review.")
        return

    try:
        # Generate a unique review ID
        review_id = get_next_review_id()
        if review_id is None:
            return  # Abort if there's an issue with generating review ID

        # Insert the review into the database
        cur.execute(
            '''
            INSERT INTO HasReview (review_ID, review_text,  rating, date,  T_username,  tour_ID)
            VALUES (?, ?, ?, ?, ?, ?)
            ''',
            (review_id, review.strip(), rating , today, login_username, tour_id)
        )
        con.commit()
        sg.popup("Review submitted successfully!")
    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")

# Paid Tours Window
def window_view_paid_tours():
    past_tours, upcoming_tours = get_customer_tours()

    past_tour_list = [
        f"ID: {tour[0]} | Name: {tour[1]} | Start: {tour[2]} | End: {tour[3]} | Price: {tour[4]} | Itinerary: {tour[5]}"
        for tour in past_tours
    ]
    upcoming_tour_list = [
        f"ID: {tour[0]} | Name: {tour[1]} | Start: {tour[2]} | End: {tour[3]} | Price: {tour[4]} | Itinerary: {tour[5]}"
        for tour in upcoming_tours
    ]

    layout = [
        [sg.Text("Your Paid Tours")],
        [sg.Text("Upcoming Tours:")],
        [sg.Listbox(values=upcoming_tour_list, size=(80, 10), key="UPCOMING_TOURS")],
        [sg.Text("Past Tours:")],
        [sg.Listbox(values=past_tour_list, size=(80, 10), key="PAST_TOURS", enable_events=True)],
        [sg.Button("Leave a Review"), sg.Button("Back")]
    ]
    return sg.Window("Paid Tours", layout)

# Handle Paid Tours Logic
def handle_paid_tours():
    window = window_view_paid_tours()

    while True:
        event, values = window.read()

        if event == sg.WIN_CLOSED or event == "Back":
            break

        elif event == "Leave a Review":
            selected_tour = values["PAST_TOURS"]
            if selected_tour and "ID:" in selected_tour[0]:
                tour_id = selected_tour[0].split('|')[0].strip().replace("ID: ", "")
                tour_name = selected_tour[0].split('|')[1].strip().replace("Name: ", "")
                review_window = window_leave_review(tour_id, tour_name)

                while True:
                    review_event, review_values = review_window.read()

                    if review_event == sg.WIN_CLOSED or review_event == "Cancel":
                        review_window.close()
                        break

                    elif review_event == "Submit":
                        handle_leave_review(tour_id, review_values)
                        review_window.close()
                        break
            else:
                sg.popup("Please select a past tour to leave a review.")



# Get all tours with their overall ratings for the logged-in customer
def get_all_tours_with_reviews():
    try:
        cur.execute(
            '''
            SELECT t.tour_ID, t.tour_name, AVG(r.rating) AS avg_rating
            FROM Tour t
            LEFT JOIN HasReview r ON t.tour_ID = r.tour_ID
            GROUP BY t.tour_ID, t.tour_name
            '''
        )
        return cur.fetchall()  # Returns a list of tuples (tour_ID, tour_name, avg_rating)
    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")
        return []


# Tour Reviews Window
def window_tour_reviews():
    tours = get_all_tours_with_reviews()

    if not tours:
        sg.popup("No reviews available for your tours!")
        return None

    tour_list = [
        f"ID: {tour[0]} | Name: {tour[1]} | Average Rating: {tour[2]:.1f} stars" 
        if tour[2] is not None else f"ID: {tour[0]} | Name: {tour[1]} | No ratings yet"
        for tour in tours
    ]

    layout = [
        [sg.Text("Tours and Ratings:")],
        [sg.Listbox(values=tour_list, size=(80, 10), key="TOUR_LIST", enable_events=True)],
        [sg.Button("Back")]
    ]
    return sg.Window("Tour Reviews", layout)

# Function to fetch reviews and display them in a popup
def display_tour_reviews(tour_id, tour_name):
    try:
        cur.execute(
            '''
            SELECT u.Name, r.rating, r.review_text
            FROM HasReview r
            JOIN Traveler t ON r.T_username = t.T_username
            JOIN User u ON t.T_username = u.Username
            WHERE r.tour_ID = ?
            ''',
            (tour_id,)
        )
        reviews = cur.fetchall()

        if not reviews:
            sg.popup(f"No reviews available for the tour: {tour_name}")
            return

        review_texts = [
            f"{review[0]} left {review[1]} stars and said: {review[2]}" 
            for review in reviews
        ]
        sg.popup_scrolled(
            *review_texts,
            title=f"Reviews for {tour_name}",
            size=(60, 20)
        )
    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")

# Get all tours with their average ratings sorted from best to worst
def get_sorted_tours():
    try:
        cur.execute(
            '''
            SELECT t.tour_ID, t.tour_name, t.Sdate, t.Edate, AVG(r.rating) AS avg_rating
            FROM Tour t
            LEFT JOIN HasReview r ON t.tour_ID = r.tour_ID
            GROUP BY t.tour_ID, t.tour_name, t.Sdate, t.Edate
            ORDER BY avg_rating DESC
            '''
        )
        return cur.fetchall()  # Returns a list of tuples (tour_ID, tour_name, Sdate, Edate, avg_rating)
    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")
        return []

# Update Tours Window
def window_update_tours():
    tours = get_sorted_tours()

    tour_list = [
        f"ID: {tour[0]} | Name: {tour[1]} | Start: {tour[2]} | End: {tour[3]} | Avg Rating: {tour[4]:.1f}" 
        if tour[4] is not None else f"ID: {tour[0]} | Name: {tour[1]} | Start: {tour[2]} | End: {tour[3]} | No ratings"
        for tour in tours
    ]

    layout = [
        [sg.Text("Update Tours (Filtered and Sorted by Ratings)")],
        [sg.Text("Filter by date range:"), sg.Combo(["Last 1 month", "Last 2 months", "Last 3 months"], key="FILTER")],
        [sg.Button("Apply Filter"), sg.Button("Reset")],
        [sg.Listbox(values=tour_list, size=(80, 10), key="TOUR_LIST", enable_events=True)],
        [sg.Button("Back")]
    ]
    return sg.Window("Update Tours", layout)

# Apply filtering logic based on date range
def apply_filter(filter_option):
    try:
        today = datetime.now()
        if filter_option == "Last 1 month":
            start_date = (today - timedelta(days=30)).strftime('%Y-%m-%d')
        elif filter_option == "Last 2 months":
            start_date = (today - timedelta(days=60)).strftime('%Y-%m-%d')
        elif filter_option == "Last 3 months":
            start_date = (today - timedelta(days=90)).strftime('%Y-%m-%d')
        else:
            return get_sorted_tours()

        cur.execute(
            '''
            SELECT t.tour_ID, t.tour_name, t.Sdate, t.Edate, AVG(r.rating) AS avg_rating
            FROM Tour t
            LEFT JOIN HasReview r ON t.tour_ID = r.tour_ID
            WHERE t.Sdate >= ?
            GROUP BY t.tour_ID, t.tour_name, t.Sdate, t.Edate
            ORDER BY avg_rating DESC
            ''',
            (start_date,)
        )
        return cur.fetchall()
    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")
        return []

# Tour Details Window for Admin
def window_tour_details_admin(tour_id, tour_name):
    layout = [
        [sg.Text(f"Tour Details: {tour_name}")],
        [sg.Button("Guide Salary"), sg.Button("Change Capacity"), sg.Button("Delete Tour")],
        [sg.Text("Reviews:")],
        [sg.Multiline(size=(60, 15), key="REVIEWS", disabled=True)],
        [sg.Button("Back")]
    ]

    # Fetch and display reviews
    try:
        cur.execute(
            '''
            SELECT u.Name, r.rating, r.review_text
            FROM HasReview r
            JOIN Traveler t ON r.T_username = t.T_username
            JOIN User u ON t.T_username = u.Username
            WHERE r.tour_ID = ?
            ''',
            (tour_id,)
        )
        reviews = cur.fetchall()
        review_texts = "\n\n".join(
            f"{review[0]} left {review[1]} stars and said: {review[2]}" for review in reviews
        )
    except sqlite3.Error as e:
        review_texts = f"Error fetching reviews: {e}"

    return sg.Window("Tour Details (Admin)", layout, finalize=True), review_texts

# Change Capacity Popup
def popup_change_capacity(tour_id):
    try:
        cur.execute("SELECT max_cap, current_cap FROM Tour WHERE tour_ID = ?", (tour_id,))
        result = cur.fetchone()
        if result:
            max_cap, current_cap = result
        else:
            sg.popup_error("Tour not found!")
            return

        layout = [
            [sg.Text(f"Current Max Capacity: {max_cap}")],
            [sg.Text(f"Current Capacity: {current_cap}")],
            [sg.Text("New Max Capacity:"), sg.Input(key="NEW_MAX_CAP")],
            [sg.Button("Submit"), sg.Button("Cancel")]
        ]
        window = sg.Window("Change Capacity", layout)
        event, values = window.read()
        window.close()

        if event == "Submit":
            new_max_cap = values.get("NEW_MAX_CAP")
            if new_max_cap and new_max_cap.isdigit():
                cur.execute("UPDATE Tour SET max_cap = ? WHERE tour_ID = ?", (int(new_max_cap), tour_id))
                con.commit()
                sg.popup("Max capacity updated successfully!")
            else:
                sg.popup("Please enter a valid number!")

    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")

# Fetch tour guides and their salaries for a specific tour
def get_tour_guides_and_salaries(tour_id):
    try:
        cur.execute(
            '''
            SELECT g.TG_username, g.salary
            FROM TourGuide g
            JOIN Assign a ON g.TG_username = a.TG_username
            WHERE a.tour_ID = ?
            ''',
            (tour_id,)
        )
        return cur.fetchall()  # Returns a list of tuples (TG_username, salary)
    except sqlite3.Error as e:
        sg.popup_error(f"Database error: {e}")
        return []

# Guide Salary Window
def window_guide_salary(tour_id):
    guides = get_tour_guides_and_salaries(tour_id)

    if not guides:
        sg.popup("No tour guides assigned to this tour.")
        return None

    guide_list = [
        f"Username: {guide[0]} | Salary: {guide[1]}" for guide in guides
    ]

    layout = [
        [sg.Text("Tour Guides and Salaries:")],
        [sg.Listbox(values=guide_list, size=(60, 10), key="GUIDE_LIST", enable_events=True)],
        [sg.Button("Submit"), sg.Button("Back")]
    ]
    return sg.Window("Guide Salary", layout), guides

# Popup to edit a guide's salary
def popup_edit_salary(username, current_salary):
    layout = [
        [sg.Text(f"Edit Salary for: {username}")],
        [sg.Text(f"Current Salary: {current_salary}")],
        [sg.Text("New Salary:"), sg.Input(key="NEW_SALARY")],
        [sg.Button("Submit"), sg.Button("Cancel")]
    ]
    window = sg.Window("Edit Salary", layout)
    event, values = window.read()
    window.close()

    if event == "Submit":
        new_salary = values.get("NEW_SALARY")
        if new_salary and new_salary.isdigit():
            return int(new_salary)
        else:
            sg.popup("Invalid salary. Please enter a valid number.")
            
    return None

# Handle Guide Salary Logic
def handle_guide_salary(tour_id):
    salary_window, guides = window_guide_salary(tour_id)
    if not salary_window:
        return

    # Track updated salaries
    updated_salaries = {guide[0]: guide[1] for guide in guides}

    while True:
        event, values = salary_window.read()

        if event == sg.WIN_CLOSED or event == "Back":
            salary_window.close()
            break

        elif event == "GUIDE_LIST":
            selected_guide = values["GUIDE_LIST"]
            if selected_guide and "Username:" in selected_guide[0]:
                username = selected_guide[0].split('|')[0].replace("Username: ", "").strip()
                current_salary = selected_guide[0].split('|')[1].replace("Salary: ", "").strip()

                # Popup to edit salary
                new_salary = popup_edit_salary(username, current_salary)
                if new_salary is not None:
                    updated_salaries[username] = new_salary
                    updated_list = [
                        f"Username: {guide[0]} | Salary: {updated_salaries[guide[0]]}" for guide in guides
                    ]
                    salary_window["GUIDE_LIST"].update(values=updated_list)

        elif event == "Submit":
            # Update all changed salaries in the database
            try:
                for username, new_salary in updated_salaries.items():
                    cur.execute("UPDATE TourGuide SET salary = ? WHERE TG_username = ?", (new_salary, username))
                con.commit()
                sg.popup("Salaries updated successfully!")
            except sqlite3.Error as e:
                sg.popup_error(f"Database error: {e}")

            salary_window.close()
            break

# Delete Tour Logic
def handle_delete_tour(tour_id, tour_name):
    # First confirmation popup
    confirm = sg.popup_yes_no(f"Are you sure you want to delete the tour: {tour_name}?")
    if confirm == "Yes":
        # Second confirmation popup
        double_confirm = sg.popup_yes_no(f"This action cannot be undone.\nDo you really want to delete '{tour_name}'?")
        if double_confirm == "Yes":
            try:
                # Delete the tour from related tables first to maintain referential integrity
                cur.execute("DELETE FROM HasReview WHERE tour_ID = ?", (tour_id,))
                cur.execute("DELETE FROM Assign WHERE tour_ID = ?", (tour_id,))
                cur.execute("DELETE FROM Transit WHERE tour_ID = ?", (tour_id,))
                cur.execute("DELETE FROM Accomodate WHERE tour_ID = ?", (tour_id,))
                cur.execute("DELETE FROM Purchase WHERE tour_ID = ?", (tour_id,))
                
                # Finally, delete the tour itself
                cur.execute("DELETE FROM Tour WHERE tour_ID = ?", (tour_id,))
                con.commit()
                sg.popup(f"Tour '{tour_name}' has been deleted successfully!")
                return True
            except sqlite3.Error as e:
                sg.popup_error(f"Database error while deleting tour: {e}")
                return False
    return False




In [4]:
# Main Program Logic
def main():
    global login_username
    global login_user_type

    window = window_login()
    assigned_transport = {}
    assigned_hotels = []
    assigned_guides = [] 

    while True:
        event, values = window.read()
        if event == sg.WIN_CLOSED:
            break
        elif event == 'Sign up as Customer':
            window.close()
            window = window_sign_up()
        elif event == 'Register':
            new_window = handle_sign_up(window, values)
            # If registration is successful, return to login window
            if new_window:  
                window = new_window
        elif event == 'Cancel':
            window.close()
            window = window_login()
        elif event == 'Login':
            username, password = values['username'], values['Password']
            if not username or not password:
                sg.popup("Username and Password cannot be left blank!")
                continue
            cur.execute("SELECT username FROM User WHERE username = ? AND password = ?", (username, password))
            result = cur.fetchone()
            if result:
                # Save the logged-in username
                login_username = username  
                cur.execute("SELECT a_username FROM Admin WHERE a_username = ?", (username,))
                admin_check = cur.fetchone()
                if admin_check:
                    sg.popup("Login successful! Welcome, Admin.")
                    login_user_type = "Admin"
                    window.close()
                    window = window_admin()
                else:
                    cur.execute("SELECT TG_username FROM TourGuide WHERE TG_username = ?", (username,))
                    t_check = cur.fetchone()
                    if t_check:
                        sg.popup("Login successful! Welcome, Tour Guide.")
                        login_user_type = "Guide"
                        window.close()
                        window = window_guide_tours(username) 
                    else:
                        cur.execute("SELECT T_username FROM Traveler WHERE T_username = ?", (username,))
                        customer_check = cur.fetchone()
                        if customer_check:
                            sg.popup("Login successful! Welcome, Customer.")
                            login_user_type = "Customer"
                            window.close()
                            window = window_customer_profile()
                        else:
                            sg.popup("Invalid username or password.")
            else:
                sg.popup("Invalid username or password.")
                
        elif event == 'Add Information':
            window.close()
            window = window_add_information()
        elif event == 'Add Credit Card Information':
            window.close()
            window = window_add_credit_card()
        elif event == 'Save':
            handle_add_credit_card(values)
            window.close()
            window = window_add_information()
            
        elif event == 'Add Contact No':
            add_contact_no_popup()
            
        elif event == 'Go Back':
            window.close()
            window = window_add_information()
            
        elif event == 'View Paid Tours':
            window.close()
            window = window_view_paid_tours()
            
        elif event == 'Back':
            if login_user_type == "Customer":
                window.close()
                window = window_customer_profile()
            elif login_user_type == "Guide":
                window.close()
                window = window_guide_tours(login_username)
                
        elif event == 'Purchase Tours':
            window.close()
            window = window_purchase_tours()
        elif event == 'Search':
            start_date = values.get('Sdate')
            end_date = values.get('Edate')
            if not validate_dates(start_date, end_date):
                continue

            tours = get_upcoming_tours(start_date, end_date)
            window['TOUR_LIST'].update(tours)

        elif event == 'TOUR_LIST':
            selected_tour = values['TOUR_LIST']
            if selected_tour and "ID:" in selected_tour[0]:
                tour_id = selected_tour[0].split('|')[0].strip() 
                window.close()
                window = window_tour_details_purchase(tour_id)
            else:
                sg.popup("Please select a tour from the list!")

        elif event == 'Pay for Tour':
            selected_tour = values['TOUR_LIST']
            if selected_tour and "ID:" in selected_tour[0]:
                tour_id = selected_tour[0].split('|')[0].strip()
                if pay_for_tour(tour_id): 
                    start_date = values.get('Sdate')
                    end_date = values.get('Edate')
                    window['TOUR_LIST'].update(get_upcoming_tours(start_date, end_date))
            else:
                sg.popup("Please select a valid tour to pay for!")

        elif event == 'Customer Information':
            selected_tour = values['GUIDE_TOUR_LIST']
            if selected_tour and "ID:" in selected_tour[0]:
                tour_id = selected_tour[0].split('|')[0].strip().replace('ID: ', '')
                customer_info = get_customer_information(tour_id)
                window.close()
                window = window_customer_information(customer_info)
            else:
                sg.popup("Please select a valid tour from the list!")
        elif event== 'Back':
            window.close()
            window= window_guide_tours(login_username)
        
        elif event == 'Create a tour':
            assigned_transport = {} 
            assigned_hotels = []  
            assigned_guides = []  
            window.close()
            window = window_add_tour()

        elif event == 'Select Transportation Options':
            try:
                start_date = datetime.strptime(values['Sdate'], '%Y-%m-%d')
                end_date = datetime.strptime(values['Edate'], '%Y-%m-%d')
                days = (end_date - start_date).days + 1
                assigned_transport = handle_add_transport(days)
            except ValueError:
                sg.popup('Invalid date format! Please use YYYY-MM-DD.')
                

        elif event == 'Add Hotels':
            start_date = values.get('Sdate')
            end_date = values.get('Edate')
            if not validate_dates(start_date, end_date):
                continue
                
            assigned_hotels = handle_add_hotel()

        elif event == 'Tour Guide':
            start_date = values.get('Sdate')
            end_date = values.get('Edate')
            if not validate_dates(start_date, end_date):
                continue
            selected_guides = window_add_tour_guides(start_date, end_date)
            if selected_guides:
                assigned_guides = selected_guides  # Update assigned_guides

        elif event == 'Add Tour':
            if not assigned_guides or len(assigned_guides) == 0:
                sg.popup("Tour guide details must be provided before adding the tour!")
            elif not assigned_transport or len(assigned_transport) == 0:
                sg.popup("Transportation details must be provided before adding the tour!")
            elif not assigned_hotels or len(assigned_hotels) == 0:
                sg.popup("Hotel details must be provided before adding the tour!")
            else:
                new_window = button_add_tour(window, values, assigned_transport, assigned_hotels, assigned_guides)
                if new_window: 
                    window = new_window

        elif event == 'Return to Main Page':
            window.close()
            window = window_admin()
        elif event == 'View tours':
            window.close()
            window = window_view_tours()
        elif event == 'Back to Main Page':
            window.close()
            window = window_admin()
        elif event == 'Logout':
            window.close()
            window = window_login()

        # Customer functionality
        elif event == 'View Paid Tours':
            window.close()
            handle_paid_tours()
            window = window_customer_profile()

        elif event == 'Leave a Review':
            selected_tour = values["PAST_TOURS"]
            if selected_tour and "ID:" in selected_tour[0]:
                tour_id = selected_tour[0].split('|')[0].strip().replace("ID: ", "")
                tour_name = selected_tour[0].split('|')[1].strip().replace("Name: ", "")
                review_window = window_leave_review(tour_id, tour_name)

                while True:
                    review_event, review_values = review_window.read()
                    if review_event == sg.WIN_CLOSED or review_event == "Cancel":
                        review_window.close()
                        break
                    elif review_event == "Submit":
                        handle_leave_review(tour_id, review_values)
                        review_window.close()
                        break
            else:
                sg.popup("Please select a past tour to leave a review.")

        elif event == 'Back':
            if login_user_type == "Customer":
                window.close()
                window = window_customer_profile()
            elif login_user_type == "Guide":
                window.close()
                window = window_guide_tours(login_username)

        elif event == 'Tour Reviews':
            window.close()
            review_window = window_tour_reviews()
            if review_window:
                while True:
                    review_event, review_values = review_window.read()
                    if review_event == sg.WIN_CLOSED or review_event == "Back":
                        review_window.close()
                        break

                    elif review_event == "TOUR_LIST":
                        selected_tour = review_values["TOUR_LIST"]
                        if selected_tour and "ID:" in selected_tour[0]:
                            tour_id = selected_tour[0].split('|')[0].strip().replace("ID: ", "")
                            tour_name = selected_tour[0].split('|')[1].strip().replace("Name: ", "")
                            display_tour_reviews(tour_id, tour_name)

            window = window_customer_profile()
        elif event == "Update Tours":
            window.close()
            update_window = window_update_tours()

            while True:
                update_event, update_values = update_window.read()
                if update_event == sg.WIN_CLOSED or update_event == "Back":
                    update_window.close()
                    break

                elif update_event == "Apply Filter":
                    filter_option = update_values["FILTER"]
                    filtered_tours = apply_filter(filter_option)
                    update_window["TOUR_LIST"].update(
                        [
                            f"ID: {tour[0]} | Name: {tour[1]} | Start: {tour[2]} | End: {tour[3]} | Avg Rating: {tour[4]:.1f}"
                            if tour[4] is not None else f"ID: {tour[0]} | Name: {tour[1]} | Start: {tour[2]} | End: {tour[3]} | No ratings"
                            for tour in filtered_tours
                        ]
                    )
                elif update_event == "Reset":
                    all_tours = get_sorted_tours()
                    update_window["TOUR_LIST"].update(
                        [
                            f"ID: {tour[0]} | Name: {tour[1]} | Start: {tour[2]} | End: {tour[3]} | Avg Rating: {tour[4]:.1f}"
                            if tour[4] is not None else f"ID: {tour[0]} | Name: {tour[1]} | Start: {tour[2]} | End: {tour[3]} | No ratings"
                            for tour in all_tours
                        ]
                    )

                elif update_event == "TOUR_LIST" and update_values["TOUR_LIST"]:
                    selected_tour = update_values["TOUR_LIST"][0]
                    tour_id = selected_tour.split("|")[0].replace("ID: ", "").strip()
                    tour_name = selected_tour.split("|")[1].replace("Name: ", "").strip()

                    details_window, review_texts = window_tour_details_admin(tour_id, tour_name)
                    details_window["REVIEWS"].update(review_texts)

                    while True:
                        detail_event, _ = details_window.read()
                        if detail_event == sg.WIN_CLOSED or detail_event == "Back":
                            details_window.close()
                            break
                        elif detail_event == "Change Capacity":
                            popup_change_capacity(tour_id)
                            
                        elif detail_event == "Guide Salary":
                            handle_guide_salary(tour_id)
                            
                        elif detail_event == "Delete Tour":
                            if handle_delete_tour(tour_id, tour_name):
                                details_window.close()
                                break
            window = window_admin()

    window.close()
    con.close()

if __name__ == "__main__":
    main()